# Decoder Training — Colab T4

Entraîne le décodeur **z → frame** avec l'encodeur LeWorldModel gelé.

**Prérequis :** avoir `lewm_best.pt` (checkpoint du world model) accessible.

**Pipeline :**
1. Check GPU
2. Clone repo
3. Upload `lewm_best.pt`
4. Générer le dataset (ou skip si déjà présent)
5. Entraîner le décodeur
6. Plot loss curves
7. Download `decoder_best.pt`

> Runtime → Change runtime type → **T4 GPU** avant de lancer.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 3. Install dependencies

In [ ]:
!pip install -q Pillow matplotlib numpy
print('Dependencies ready.')

## 4. Upload `lewm_best.pt`

Upload le checkpoint du world model. Deux options :

**Option A — upload direct** (cellule ci-dessous)  
**Option B — Google Drive** : monte Drive et copie le fichier manuellement, puis skip cette cellule.

In [ ]:
from google.colab import files
from pathlib import Path

CKPT_DIR = 'checkpoints'
Path(CKPT_DIR).mkdir(exist_ok=True)

lewm_path = f'{CKPT_DIR}/lewm_best.pt'

if Path(lewm_path).exists():
    print(f'lewm_best.pt already present — skipping upload.')
else:
    print('Upload lewm_best.pt :')
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = f'{CKPT_DIR}/{fname}'
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'Saved → {dest}')

## 5. Generate dataset

Génère **2 000 trajectoires × 500 frames** si pas encore présent.  
~5-10 min sur Colab CPU.

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from data.generate import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=64,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

## 6. Hyperparamètres

In [ ]:
# ── Hyperparamètres décodeur ────────────────────────────────────────────────────
LEWM_CKPT   = 'checkpoints/jepa/lewm_best.pt'
CKPT_DIR    = 'checkpoints'
VIS_DIR     = 'visuals'
DATASET_DIR = 'dataset/pendulum'

SEQ_LEN     = 100    # fenêtre tirée aléatoirement dans chaque trajectoire
BATCH_SIZE  = 32
EPOCHS      = 30
LR          = 3e-4
NUM_WORKERS = 2

## 7. Setup

In [ ]:
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from models.jepa.model import LeWorldModel
from models.decoder    import Decoder
from data.dataset      import make_seq_dataloaders

device = torch.device('cuda')
print(f'Device: {torch.cuda.get_device_name(0)}')

# ── Charger l'encodeur (gelé) ───────────────────────────────────────────────────
ckpt  = torch.load(LEWM_CKPT, map_location=device, weights_only=False)
args  = ckpt.get('args', {})
encoder = LeWorldModel(
    embed_dim    = args.get('embed_dim',    128),
    hidden_dim   = args.get('hidden_dim',   512),
    lam          = args.get('lam',          0.5),
    n_proj       = args.get('n_proj',       512),
    ema_momentum = args.get('ema_momentum', 0.996),
    rollout_k    = args.get('rollout_k',    10),
).to(device)
encoder.load_state_dict(ckpt['model'], strict=False)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad_(False)
EMBED_DIM = encoder.embed_dim
print(f'Encodeur chargé (epoch={ckpt.get("epoch","?")}  val_loss={ckpt.get("val_loss",float("nan")):.5f})')

# ── Décodeur ───────────────────────────────────────────────────────────────────
decoder = Decoder(embed_dim=EMBED_DIM).to(device)
n_params = sum(p.numel() for p in decoder.parameters())
print(f'Décodeur : {n_params:,} paramètres')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Optimizer ──────────────────────────────────────────────────────────────────
optimizer = optim.AdamW(decoder.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

Path(CKPT_DIR).mkdir(exist_ok=True)
Path(VIS_DIR).mkdir(exist_ok=True)

# ── Evaluate ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(encoder, decoder, loader):
    decoder.eval()
    total = 0.0
    for frames, _ in loader:
        frames = frames.to(device)
        z      = encoder.encode(frames)
        recon  = decoder(z)
        total += nn.functional.mse_loss(recon, frames).item()
    return total / len(loader)

## 8. Training loop

In [ ]:
history  = {'train': [], 'val': []}
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    decoder.train()
    t0    = time.time()
    total = 0.0

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        with torch.no_grad():
            z = encoder.encode(frames)          # (B, T, D)
        recon = decoder(z)                      # (B, T, 3, 64, 64)
        loss  = nn.functional.mse_loss(recon, frames)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    scheduler.step()
    train_loss = total / len(train_loader)
    val_loss   = evaluate(encoder, decoder, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    improved = val_loss < best_val
    if improved:
        best_val = val_loss
        torch.save({
            'epoch':     epoch,
            'decoder':   decoder.state_dict(),
            'val_loss':  best_val,
            'embed_dim': EMBED_DIM,
        }, f'{CKPT_DIR}/decoder_best.pt')

    elapsed = time.time() - t0
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  train={train_loss:.4f}'
        f'  val={val_loss:.4f}'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

print(f'\nMeilleur val MSE : {best_val:.4f}  →  {CKPT_DIR}/decoder_best.pt')

## 9. Loss curves

In [ ]:
import matplotlib.pyplot as plt

epochs_x = range(1, len(history['train']) + 1)

fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor('#111')
ax.set_facecolor('#111')
ax.plot(epochs_x, history['train'], color='#4fc3f7', label='train MSE')
ax.plot(epochs_x, history['val'],   color='#ff8a65', label='val MSE')
ax.set_title('Decoder reconstruction loss', color='white')
ax.set_xlabel('epoch', color='white')
ax.set_ylabel('MSE', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/decoder_loss.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 10. Visualiser les reconstructions

Affiche quelques frames réelles vs reconstruites pour vérifier la qualité du décodeur.

In [ ]:
import numpy as np
from data.dataset import PendulumSeqDataset

ds = PendulumSeqDataset(DATASET_DIR, seq_len=SEQ_LEN)
frames_sample, _ = ds[0]                           # (SEQ_LEN, 3, 64, 64)
frames_sample = frames_sample.unsqueeze(0).to(device)  # (1, SEQ_LEN, 3, 64, 64)

with torch.no_grad():
    z_sample = encoder.encode(frames_sample)       # (1, SEQ_LEN, D)
    recon    = decoder(z_sample).clamp(0, 1)       # (1, SEQ_LEN, 3, 64, 64)

N_SHOW = 8
idxs   = np.linspace(0, SEQ_LEN - 1, N_SHOW, dtype=int)

fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 1.6, 3.5))
fig.patch.set_facecolor('#111')
fig.suptitle('Réel (haut)  vs  Reconstruit (bas)', color='white', fontsize=11)

for col, i in enumerate(idxs):
    for row, src in enumerate([frames_sample[0], recon[0]]):
        ax = axes[row, col]
        img = src[i].permute(1, 2, 0).cpu().numpy()
        ax.imshow(img)
        ax.axis('off')
        if row == 0:
            ax.set_title(f't={i}', color='#aaa', fontsize=7)

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/decoder_recon.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

## 11. Download checkpoint

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/decoder_best.pt')